<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-08-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-10.

Características do dataset:

- 60.000 imagens RGB
- 10 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Objetivos

Nesta atividade você irá:

- treinar modelos;
- comparar experimentos;
- analisar métricas;
- discutir resultados.


Nesta atividade utilizaremos MLflow para:

- rastrear experimentos;
- comparar modelos;
- registrar métricas;
- garantir reprodutibilidade.

In [2]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import time

In [4]:
mlflow.set_experiment(
    "assignment"
)

<Experiment: artifact_location='file:///c:/cesar/ml/atividade-04-deep-learning-i-helotanaka/notebooks/mlruns/1', creation_time=1779320615240, experiment_id='1', last_update_time=1779320615240, lifecycle_stage='active', name='assignment', tags={}, trace_location=None, workspace='default'>

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-10 utilizando `tensorflow.keras.datasets.cifar10.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

**Solução**:

In [5]:
from tensorflow.keras.datasets import cifar10
from sklearn.model_selection import train_test_split

def load_data(seed=42):
    (X_train_full, y_train_full), (X_test, y_test) = cifar10.load_data()

    X_train_full = X_train_full.reshape(X_train_full.shape[0], -1)

    X_train_full = X_train_full.astype("float32") / 255.0

    y_train_full = y_train_full.ravel()

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full,
        test_size=0.2, random_state=seed, stratify=y_train_full
    )

    return X_train, X_val, y_train, y_val

X_train, X_val, y_train, y_val = load_data(seed=42)
print(f"X_train: {X_train.shape}, X_val: {X_val.shape}")
print(f"y_train: {y_train.shape}, y_val: {y_val.shape}")

X_train: (40000, 3072), X_val: (10000, 3072)
y_train: (40000,), y_val: (10000,)


1. O formato original das imagens é (32, 32, 3), representando altura, largura e canais RGB.

2. Após o flatten, cada imagem possui 3072 features.

3. O flatten é necessário porque a MLP recebe vetores unidimensionais como entrada, e não matrizes 2D/3D de imagens.

4. A normalização coloca os pixels em uma escala menor (0 a 1), o que ajuda o treinamento a convergir mais rápido e de forma mais estável.

# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**:

In [6]:
from sklearn.neural_network import MLPClassifier

def train_mlp(X_train, y_train, activation, hidden_layers, learning_rate, seed,
              max_iter=50, batch_size=128):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        random_state=seed,
        max_iter=max_iter,
        batch_size=batch_size,
        verbose=False
    )
    model.fit(X_train, y_train)
    return model

1. 3072 × h + h (pesos + biases)

2. A função da ReLU é introduzir não-linearidade no modelo, permitindo que a rede aprenda padrões mais complexos. Ela retorna 0 para valores negativos e mantém valores positivos.

3. MLPs possuem muitos parâmetros com imagens porque cada pixel é conectado a todos os neurônios da próxima camada. Como imagens possuem milhares de pixels, o número de conexões cresce muito rapidamente.

# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

**Solução**:

In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(model, X_val, y_val):
    y_pred = model.predict(X_val)

    metrics = {
        "accuracy":  accuracy_score(y_val, y_pred),
        "precision": precision_score(y_val, y_pred, average="weighted", zero_division=0),
        "recall":    recall_score(y_val, y_pred, average="weighted", zero_division=0),
        "f1_score":  f1_score(y_val, y_pred, average="weighted", zero_division=0),
    }

    print(pd.DataFrame([metrics]).to_string(index=False))
    return metrics

1. A accuracy representa a proporção total de previsões corretas feitas pelo modelo.

2. Precision mede quantas previsões positivas estavam corretas, enquanto recall mede quantos exemplos positivos reais foram encontrados pelo modelo.

3. O f1-score é importante quando existe desbalanceamento de classes ou quando precision e recall são igualmente importantes, pois ele combina os dois em uma única métrica.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**:

In [8]:
def run_experiment(X_train, y_train, X_val, y_val,
                   activation, hidden_layers, learning_rate, seed,
                   max_iter=25, batch_size=128, run_name=None):

    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("activation",    activation)
        mlflow.log_param("hidden_layers", str(hidden_layers))
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("max_iter",      max_iter)
        mlflow.log_param("batch_size",    batch_size)

        start = time.time()
        model = train_mlp(X_train, y_train, activation, hidden_layers,
                          learning_rate, seed, max_iter, batch_size)
        training_time = time.time() - start

        metrics = evaluate(model, X_val, y_val)

        mlflow.log_metrics(metrics)
        mlflow.log_metric("training_time", training_time)

        print(f"[{run_name}] accuracy={metrics['accuracy']:.4f}  tempo={training_time:.1f}s")

    return model, metrics

# Experimento baseline (Q4)
run_experiment(
    X_train, y_train, X_val, y_val,
    activation="relu", hidden_layers=(128, 64), learning_rate=0.01, seed=42,
    run_name="baseline"
)

 accuracy  precision  recall  f1_score
    0.413   0.412703   0.413  0.407677
[baseline] accuracy=0.4130  tempo=181.7s


(MLPClassifier(batch_size=128, hidden_layer_sizes=(128, 64),
               learning_rate_init=0.01, max_iter=25, random_state=42),
 {'accuracy': 0.413,
  'precision': 0.4127033532806391,
  'recall': 0.413,
  'f1_score': 0.40767714748013084})

1. Baseline (relu, (128,64), lr=0.01) — único experimento rodado, com accuracy de 41.3%.

2. Não há comparação ainda para afirmar estabilidade, mas lr=0.01 com ReLU tende a ser estável — sem oscilações extremas.

3. O rastreamento permite comparar experimentos de forma organizada, reproduzir resultados e identificar quais hiperparâmetros mais impactam o desempenho — sem depender de anotações manuais.

# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**:

In [9]:
activations = ["logistic", "tanh", "relu"]
resultados_q5 = {}

for act in activations:
    _, metrics = run_experiment(
        X_train, y_train, X_val, y_val,
        activation=act, hidden_layers=(128, 64), learning_rate=0.001, seed=42,
        run_name=f"activation_{act}"
    )
    resultados_q5[act] = metrics

df_q5 = pd.DataFrame(resultados_q5).T
print("\nComparação das funções de ativação:")
print(df_q5)

 accuracy  precision  recall  f1_score
   0.4838   0.500799  0.4838  0.482701
[activation_logistic] accuracy=0.4838  tempo=125.1s
 accuracy  precision  recall  f1_score
   0.4286   0.459148  0.4286  0.430381
[activation_tanh] accuracy=0.4286  tempo=103.7s
 accuracy  precision  recall  f1_score
   0.4909   0.500226  0.4909  0.489203
[activation_relu] accuracy=0.4909  tempo=139.1s

Comparação das funções de ativação:
          accuracy  precision  recall  f1_score
logistic    0.4838   0.500799  0.4838  0.482701
tanh        0.4286   0.459148  0.4286  0.430381
relu        0.4909   0.500226  0.4909  0.489203


1. ReLU, maior accuracy (49.1%) e F1 (0.489).

2. ReLU, melhor resultado com tempo razoável. Logistic foi próxima mas mais lenta.

3. Sim, mas moderadas. ReLU e logistic ficaram próximas, enquanto tanh ficou ~6 p.p. abaixo, sugerindo que não se adaptou bem a essa configuração.

4. Evita vanishing gradient para valores positivos, é computacionalmente simples e converge bem em redes profundas.

# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**:

In [10]:
architectures = [(32,), (64,), (128, 64), (256, 128)]
resultados_q6 = {}

for arch in architectures:
    _, metrics = run_experiment(
        X_train, y_train, X_val, y_val,
        activation="relu", hidden_layers=arch, learning_rate=0.001, seed=42,
        run_name=f"arch_{arch}"
    )
    resultados_q6[str(arch)] = metrics

df_q6 = pd.DataFrame(resultados_q6).T
df_q6.index.name = "arquitetura"
print("\nComparação das arquiteturas:")
print(df_q6)

 accuracy  precision  recall  f1_score
   0.3795   0.368703  0.3795  0.362668
[arch_(32,)] accuracy=0.3795  tempo=46.0s
 accuracy  precision  recall  f1_score
   0.3774   0.369356  0.3774  0.364569
[arch_(64,)] accuracy=0.3774  tempo=50.1s
 accuracy  precision  recall  f1_score
   0.4909   0.500226  0.4909  0.489203
[arch_(128, 64)] accuracy=0.4909  tempo=137.8s
 accuracy  precision  recall  f1_score
   0.5088   0.515903  0.5088  0.504539
[arch_(256, 128)] accuracy=0.5088  tempo=272.7s

Comparação das arquiteturas:
             accuracy  precision  recall  f1_score
arquitetura                                       
(32,)          0.3795   0.368703  0.3795  0.362668
(64,)          0.3774   0.369356  0.3774  0.364569
(128, 64)      0.4909   0.500226  0.4909  0.489203
(256, 128)     0.5088   0.515903  0.5088  0.504539


1. Sim, neste caso redes maiores melhoraram de 37.9% em (32,) até 50.9% em (256,128). Mas o ganho não é linear e o custo cresce muito.

2. (128, 64), 49.1% de accuracy em 128s, contra 50.9% da (256,128) que levou 351s para ganhar apenas 2 p.p.

3. Não é possível afirmar com certeza sem ver a accuracy de treino, mas é esperado em (256,128) dado o tamanho da rede.

4. (256, 128), 351s, quase 3x mais lento que (128,64).

# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

In [11]:
learning_rates = [0.1, 0.01, 0.001]
resultados_q7 = {}

for lr in learning_rates:
    _, metrics = run_experiment(
        X_train, y_train, X_val, y_val,
        activation="relu", hidden_layers=(128, 64), learning_rate=lr, seed=42,
        run_name=f"lr_{lr}"
    )
    resultados_q7[lr] = metrics

df_q7 = pd.DataFrame(resultados_q7).T
df_q7.index.name = "learning_rate"
print("\nComparação dos learning rates:")
print(df_q7)

 accuracy  precision  recall  f1_score
      0.1       0.01     0.1  0.018182
[lr_0.1] accuracy=0.1000  tempo=256.1s
 accuracy  precision  recall  f1_score
    0.413   0.412703   0.413  0.407677
[lr_0.01] accuracy=0.4130  tempo=176.7s
 accuracy  precision  recall  f1_score
   0.4909   0.500226  0.4909  0.489203
[lr_0.001] accuracy=0.4909  tempo=134.4s

Comparação dos learning rates:
               accuracy  precision  recall  f1_score
learning_rate                                       
0.100            0.1000   0.010000  0.1000  0.018182
0.010            0.4130   0.412703  0.4130  0.407677
0.001            0.4909   0.500226  0.4909  0.489203


1. 0.001,accuracy de 49.1% e F1 de 0.489, melhor entre os três.

2. 0.1, accuracy de 10% (acaso para 10 classes), F1 de 0.018, claramente não convergiu.

3. Os pesos são atualizados com passos grandes demais, o modelo oscila e não consegue convergir como visto com lr=0.1 que ficou no acaso mesmo após 306s de treino.

4. O modelo aprende devagar mas de forma mais precisa, convergindo para mínimos melhores lr=0.001 foi o mais lento em tempo por época mas entregou o melhor resultado.

# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

---
**Comportamento da loss:** A loss caiu consistentemente conforme o learning rate diminuiu e as arquiteturas cresceram. Com lr=0.1 a loss oscilou sem convergir; com lr=0.001 a descida foi suave e estável.

**Impacto do learning rate:** Foi o fator de maior impacto isolado. lr=0.1 colapsou para acaso (10%), lr=0.01 atingiu 41.3%, lr=0.001 chegou a 49.1% — mostrando que passos menores permitem convergência mais precisa.

**Impacto da arquitetura:** Arquiteturas maiores ajudaram, mas com retorno decrescente. (128,64) entregou 49.1% em 128s; (256,128) entregou 50.9% em 351s — ganho de ~2 p.p. custando quase 3x mais tempo.

**Impacto das funções de ativação:** ReLU foi a melhor (49.1%), seguida de logistic (48.4%) e tanh (42.9%). A diferença foi moderada, indicando que o gargalo principal é a própria limitação da MLP para imagens, não a ativação.

**Comportamento do treinamento:** Com max_iter limitado, todos os modelos ainda tinham margem para melhorar. Modelos maiores com lr baixo precisam de mais épocas para convergir completamente.

**Limitações da MLP:** O flatten destrói a estrutura espacial da imagem — pixels vizinhos que formam bordas e texturas perdem sua relação de vizinhança. Além disso, sem compartilhamento de pesos, o número de parâmetros cresce de forma explosiva com o tamanho da entrada.

**Backpropagation e aprendizado:** O backpropagation calcula o gradiente do erro em relação a cada peso via regra da cadeia, propagando o sinal de erro da saída até a entrada. Isso permite que cada peso seja ajustado na direção que reduz a loss, viabilizando o aprendizado em redes com múltiplas camadas.

---

1. **Melhor configuração:** relu + (256,128) + lr=0.001 — accuracy de 50.9%, mas (128,64) + lr=0.001 oferece melhor tradeoff com 49.1% em muito menos tempo.
2. **Principais dificuldades:** Sensibilidade extrema ao learning rate, alto custo computacional das arquiteturas maiores, e convergência incompleta dentro do limite de iterações.
3. **Limitações para imagens:** O flatten rompe a correlação espacial entre pixels. A MLP trata cada pixel de forma independente, sem capturar padrões locais como bordas e texturas — o que CNNs fazem naturalmente via convolução.
4. **Backpropagation:** Propaga o erro camada a camada, calculando a contribuição de cada peso para o erro total e ajustando-os para minimizar a loss. Sem ele, treinar redes profundas seria inviável.